# 🧹 Notebook 02 — Data Cleaning + SQL Database
**Bluestock Fintech | Day 2**

In [ ]:
import pandas as pd, numpy as np, sqlite3, matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = Path('..').resolve()
RAW  = BASE/'data'/'raw'; PROC = BASE/'data'/'processed'
DB   = BASE/'data'/'db'/'bluestock_mf.db'
print("Paths initialised ✓")

## 1. Clean NAV History — Forward Fill Holidays

In [ ]:
df_nav = pd.read_csv(RAW/'02_nav_history.csv', parse_dates=['date'])
print(f"Raw shape: {df_nav.shape}")

# Sort, deduplicate
df_nav = df_nav.sort_values(['amfi_code','date']).drop_duplicates(['amfi_code','date'])

# Reindex to full date range & ffill (handles weekends/holidays)
full_date_range = pd.date_range(df_nav['date'].min(), df_nav['date'].max(), freq='D')
cleaned_dfs = []
for code, grp in df_nav.groupby('amfi_code'):
    grp = grp.set_index('date').reindex(full_date_range)
    grp['nav'] = grp['nav'].ffill()
    grp['amfi_code'] = code
    grp.index.name = 'date'
    cleaned_dfs.append(grp.reset_index())

df_nav_clean = pd.concat(cleaned_dfs, ignore_index=True)
df_nav_clean = df_nav_clean[df_nav_clean['nav'].notna()]
df_nav_clean['nav'] = df_nav_clean['nav'].round(4)
print(f"Cleaned shape: {df_nav_clean.shape}")
print(f"Null NAV remaining: {df_nav_clean['nav'].isna().sum()}")
df_nav_clean.to_csv(PROC/'clean_nav.csv', index=False)
print("clean_nav.csv saved ✓")

## 2. Clean Investor Transactions

In [ ]:
df_tx = pd.read_csv(RAW/'08_investor_transactions.csv', parse_dates=['transaction_date'])
print(f"Raw transactions: {len(df_tx):,}")

# Standardise transaction_type
df_tx['transaction_type'] = df_tx['transaction_type'].str.strip().str.title()
valid_types = ['Sip','Lumpsum','Redemption']
df_tx = df_tx[df_tx['transaction_type'].isin(valid_types)]

# Validate amounts
df_tx = df_tx[df_tx['amount_inr'] > 0]

# KYC status
df_tx['kyc_status'] = df_tx['kyc_status'].str.strip()

# Fix city_tier
df_tx['city_tier'] = df_tx['city_tier'].str.upper().str.strip()
df_tx = df_tx[df_tx['city_tier'].isin(['T30','B30'])]

print(f"Clean transactions: {len(df_tx):,}")
print(f"Missing values:\n{df_tx.isnull().sum()[df_tx.isnull().sum()>0]}")
df_tx.to_csv(PROC/'clean_transactions.csv', index=False)
print("clean_transactions.csv saved ✓")

## 3. Clean Scheme Performance

In [ ]:
df_perf = pd.read_csv(RAW/'07_scheme_performance.csv')

# Ensure numeric
numeric_cols = ['return_1yr_pct','return_3yr_pct','return_5yr_pct',
                'alpha','beta','sharpe_ratio','sortino_ratio',
                'std_dev_ann_pct','max_drawdown_pct']
for col in numeric_cols:
    if col in df_perf.columns:
        df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')

# Validate expense ratio range
if 'expense_ratio_pct' in df_perf.columns:
    mask = df_perf['expense_ratio_pct'].between(0.01, 3.0)
    print(f"Expense ratio out of range: {(~mask).sum()} rows")

print(f"Negative Sharpe funds: {(df_perf.get('sharpe_ratio',pd.Series())<0).sum()}")
df_perf.to_csv(PROC/'clean_performance.csv', index=False)
print("clean_performance.csv saved ✓")
df_perf.describe()

## 4. Load & Query SQLite Database

In [ ]:
conn = sqlite3.connect(DB)
print("Connected to:", DB)

# Show all tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("\nTables in DB:")
for t in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn)['n'].iloc[0]
    print(f"  {t:30s}  {count:>8,} rows")

## 5. Run the 10 Analytical SQL Queries

In [ ]:
queries = {
    "Q1 Top funds by composite score": '''
        SELECT f.scheme_name, f.fund_house, f.sub_category,
               p.cagr_3yr_pct, p.sharpe_ratio, p.composite_score, p.score_rank
        FROM fact_performance p JOIN dim_fund f ON p.amfi_code=f.amfi_code
        ORDER BY p.score_rank LIMIT 10''',

    "Q2 Avg NAV by category (last 6 months)": '''
        SELECT f.sub_category, ROUND(AVG(n.nav_inr),2) as avg_nav, COUNT(DISTINCT n.amfi_code) as funds
        FROM fact_nav n JOIN dim_fund f ON n.amfi_code=f.amfi_code
        WHERE n.nav_date >= '2025-12-01'
        GROUP BY f.sub_category ORDER BY avg_nav DESC''',

    "Q3 AUM market share latest quarter": '''
        SELECT fund_house, aum_lakh_crore,
               ROUND(aum_lakh_crore/SUM(aum_lakh_crore) OVER()*100,2) as market_share_pct
        FROM fact_aum WHERE quarter=(SELECT MAX(quarter) FROM fact_aum)
        ORDER BY aum_lakh_crore DESC''',

    "Q4 SIP by state T30 vs B30": '''
        SELECT state, city_tier, COUNT(*) as txns,
               ROUND(SUM(amount_inr)/1e7,2) as total_crore,
               ROUND(AVG(amount_inr),0) as avg_sip
        FROM fact_transactions WHERE transaction_type='Sip'
        GROUP BY state, city_tier ORDER BY total_crore DESC LIMIT 15''',

    "Q5 Low expense + high Sharpe funds": '''
        SELECT f.scheme_name, f.expense_ratio_pct, p.sharpe_ratio, p.cagr_3yr_pct
        FROM dim_fund f JOIN fact_performance p ON f.amfi_code=p.amfi_code
        WHERE f.expense_ratio_pct < 0.50 AND p.sharpe_ratio > 0.50
        ORDER BY p.sharpe_ratio DESC''',
}

for qname, qsql in queries.items():
    print(f"\n{'='*55}")
    print(f"  {qname}")
    print('='*55)
    try:
        result = pd.read_sql(qsql.strip(), conn)
        print(result.to_string(index=False))
    except Exception as e:
        print(f"  ERROR: {e}")

conn.close()
print("\n✅ All queries executed")

## 6. Data Quality Summary

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,2,figsize=(12,4))

# Missing values heatmap
df_all = pd.read_csv(RAW/'08_investor_transactions.csv')
missing = df_all.isnull().sum().sort_values(ascending=True)
missing = missing[missing>0]
if len(missing):
    missing.plot(kind='barh', ax=axes[0], color='tomato')
    axes[0].set_title('Missing Values by Column')
else:
    axes[0].text(0.5,0.5,'No Missing Values!', ha='center', va='center', fontsize=14, color='green')
    axes[0].set_title('Data Quality — Investor Transactions')

# Transaction type split
df_tx2 = pd.read_csv(PROC/'clean_transactions.csv')
counts = df_tx2['transaction_type'].value_counts()
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2196F3','#4CAF50','#FF5722'])
axes[1].set_title('Transaction Type Distribution')

plt.tight_layout()
plt.savefig(BASE/'data'/'processed'/'nb02_quality.png',dpi=120,bbox_inches='tight')
plt.show()
print("✅ Cleaning complete. All CSVs saved to data/processed/")